# 03 — Hierarchical Surrogate Inverse Optimization

두 경로를 결합합니다.
1. **Explainable path:** desired performance → target descriptor latent → raw structural descriptors.
2. **Feasibility refinement:** generator parameters → Model 1 → Model 2 → desired performance를 직접 최소화.

첨부 `Com_Optimization_v19`의 baseline/utility/Pareto/uncertainty 철학을 Voxel 생성인자 공간으로 옮긴 버전입니다.


In [ ]:
from pathlib import Path
import sys,json,joblib
PROJECT_POINTER=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation\.ai_voxel_ml_project.json")
CODE_DIR=Path.cwd()/"Code" if (Path.cwd()/"Code").exists() else Path.cwd();sys.path.insert(0,str(CODE_DIR)) if str(CODE_DIR) not in sys.path else None
RANDOM_SEED=42
TARGET_SPEC={
 'plateau_stress':{'mode':'target','target':5.0,'scale':2.0,'weight':1.0},
 'densification_strain':{'mode':'maximize','min':0.45,'scale':0.20,'weight':1.0},
 'absorbed_energy_to_densification':{'mode':'maximize','scale':5.0,'weight':1.2},
 'peak_stress':{'mode':'minimize','max':10.0,'scale':4.0,'weight':0.6},
}
GENERATOR_FAMILIES=['latent_periodic_isotropic','latent_periodic_orthotropic','latent_stochastic'];N_DESCRIPTOR_TARGETS=3

TARGET_CURVE_CSV=None # optional CSV with columns strain,target_stress_MPa
TARGET_CURVE_WEIGHT=0.0 # e.g. 0.5~2.0 when matching an entire desired compression curve


In [ ]:
import joblib,json
from voxel_ml_common import load_contract,mark_stage
from inverse_design_engine import run_hierarchical_inverse
c=load_contract(PROJECT_POINTER);mr=Path(c['model_root']);out=Path(c['inverse_root']);out.mkdir(parents=True,exist_ok=True)
gen=joblib.load(mr/'01_gen2desc'/'gen2desc_bundle.joblib');curve=joblib.load(mr/'02_desc2curve'/'desc2curve_bundle.joblib')
target_curve=None
if TARGET_CURVE_CSV is not None:
    import pandas as pd,numpy as np
    tc=pd.read_csv(TARGET_CURVE_CSV);target_curve=np.interp(curve['strain_grid'],pd.to_numeric(tc['strain']),pd.to_numeric(tc['target_stress_MPa']))
gdf,ddf=run_hierarchical_inverse(gen,curve,TARGET_SPEC,out,GENERATOR_FAMILIES,N_DESCRIPTOR_TARGETS,RANDOM_SEED,target_curve=target_curve,curve_weight=TARGET_CURVE_WEIGHT)
mark_stage(out,'01_hierarchical_inverse','completed',[out/'optimized_descriptor_states.csv',out/'optimized_raw_structural_descriptors.csv',out/'optimized_generator_parameters.csv',out/'optimized_predicted_curves.csv'],{'candidates':len(gdf)})
display(ddf);display(gdf.head(20));print('\nDatasetFactory input:',out/'optimized_generator_parameters.csv')


### 다음 시험 단계
`00_AI_Voxel_DatasetFactory_v4.ipynb`의 새 RUN_NAME에서 `CANDIDATE_SOURCE="inverse_design"`, `INVERSE_DESIGN_CANDIDATE_FILE=...optimized_generator_parameters.csv`로 지정합니다. Imported inverse candidates는 LHS를 건너뛰고 모두 final Voxel → STL → DLP → descriptor로 검증됩니다.
